# Part 3 — Work IQ MCP

**MCP (Model Context Protocol)** exposes Work IQ as a **tool** to any MCP-capable agent (GitHub
Copilot CLI, Foundry agents, Agent Framework, …). Use MCP when your agent treats Work IQ as an
**agent-to-tool** dependency.

The 10 generic verbs:

| Read / discover | Write |
|---|---|
| `ask`, `fetch`, `search_paths`, `get_schema`, `call_function`, `list_agents` | `create_entity`, `update_entity`, `delete_entity`, `do_action` |

Key idea: **the tool surface never grows** as new data sources are added — you discover new
*paths* at runtime with `get_schema`.

In [ ]:
import sys, os, json
sys.path.append(os.path.abspath(".."))
from notebooks._shared import get_user_token, call_mcp, call_tool

token = get_user_token()
# Work IQ speaks MCP (JSON-RPC 2.0) at {gateway}/mcp and streams responses over SSE.
# call_mcp / call_tool in _shared.py handle the envelope + `data:` frame parsing.
def mcp_call(name, arguments):
    return call_tool(token, name, arguments)
print("Signed in - Work IQ MCP helper ready. Token:", token[:16], "...")

## 1. List the tools the server offers

In [ ]:
tools = call_mcp(token, "tools/list", {})
for tdef in tools["result"]["tools"]:
    req = tdef.get("inputSchema", {}).get("required", [])
    print(f"{tdef['name']:<14} required={req}")

## 2. Runtime schema discovery with `get_schema`

Discover the shape of a path before you call it — no pre-baked models.

In [ ]:
# get_schema needs operationType: one of fetch | create | update.
schema = mcp_call("get_schema", {"path": "/me/messages", "operationType": "fetch"})
print(schema["result"]["content"][0]["text"][:1200])

## 3. `fetch` — read the user's mail

In [ ]:
# fetch reads one or more entities by path via `entityUrls`.
msgs = mcp_call("fetch", {"entityUrls": ["/me/messages?$top=5&$select=subject,from,receivedDateTime"]})
for res in msgs["result"]["structuredContent"]["results"]:
    for m in res["data"].get("value", [])[:5]:
        print(m.get("receivedDateTime"), "-", m.get("subject"))

### Recap

- One compact verb set over runtime-discovered M365 paths.
- Same security model as everything else in Work IQ — delegated, permission-aware.

**Next:** Part 4 — taking action with `do_action`.